In [53]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()
llm=ChatOpenAI(model="deepseek-chat",
               api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
               temperature=0
               )

In [54]:
import requests
import json

def fetch_real_time_info(query):
    """get real-time Internet information"""
    url="https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&t_weather={query}"

    response=requests.get(url)
    data=json.loads(response.text)
    print(data)
    return data


In [55]:
from langchain_core.tools import tool

@tool
def fetch_real_time_info(query):
    """get real-time Internet information"""
    url="https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&t_weather=true"
    print("query:"+query)
    response=requests.get(url)
    data=json.loads(response.text)
    return data

In [56]:
print(f'''
name: {fetch_real_time_info.name}
description: {fetch_real_time_info.description}
arguments: {fetch_real_time_info.args}
''')


name: fetch_real_time_info
description: get real-time Internet information
arguments: {'query': {'title': 'Query'}}



In [57]:
from langgraph.prebuilt import ToolNode

tools=[fetch_real_time_info]
tool_node=ToolNode(tools) 

In [58]:
tool_node

tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'runtime': ('N/A', <class 'inspect._empty'>)}, _tools_by_name={'fetch_real_time_info': StructuredTool(name='fetch_real_time_info', description='get real-time Internet information', args_schema=<class 'langchain_core.utils.pydantic.fetch_real_time_info'>, func=<function fetch_real_time_info at 0x000001DE7FABA160>)}, _injected_args={'fetch_real_time_info': _InjectedArgs(state={}, store=None, runtime=None, all_injected_keys=set(), _optional_state_args=set())}, _handle_tool_errors=<function _default_handle_tool_errors at 0x000001DE7CDB5580>, _messages_key='messages', _wrap_tool_call=None, _awrap_tool_call=None)

In [59]:
from langchain_core.messages import AIMessage
from langgraph.runtime import Runtime
from langgraph._internal._constants import CONF, CONFIG_KEY_RUNTIME

message_with_single_tool_call=AIMessage(
    content="",
    tool_calls=[{
        "name":"fetch_real_time_info",
        "args":{"query":"小米汽车"},
        "id":"tool_call_id",
        "type":"tool_call"
    }]
)

# ✅ ToolNode._func 需要 config 和 runtime 两个注入参数。
# 独立调用(invoke)时没有 Graph 运行时，需要手动注入一个最小 Runtime 实例到 config 中。
config = {CONF: {CONFIG_KEY_RUNTIME: Runtime()}}
tool_node.invoke({"messages":[message_with_single_tool_call]}, config)

query:小米汽车


{'messages': [ToolMessage(content='{"latitude": 52.52, "longitude": 13.419998, "generationtime_ms": 0.001430511474609375, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 38.0}', name='fetch_real_time_info', tool_call_id='tool_call_id')]}

In [60]:
@tool
def get_weather(location):
    """call to get the current weather."""
    if location.lower() in ["beijing"]:
        return "北京的温度是16度，天气晴朗。"
    elif location.lower() in ["shanghai"]:
        return "上海的温度是30度，天气多云"
    else:
        return "不好意思，并未查询到具体的天气信息"

In [61]:
tools=[fetch_real_time_info,get_weather]
tool_node=ToolNode(tools)

In [62]:
tool_node

tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'runtime': ('N/A', <class 'inspect._empty'>)}, _tools_by_name={'fetch_real_time_info': StructuredTool(name='fetch_real_time_info', description='get real-time Internet information', args_schema=<class 'langchain_core.utils.pydantic.fetch_real_time_info'>, func=<function fetch_real_time_info at 0x000001DE7FABA160>), 'get_weather': StructuredTool(name='get_weather', description='call to get the current weather.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001DE7FABB880>)}, _injected_args={'fetch_real_time_info': _InjectedArgs(state={}, store=None, runtime=None, all_injected_keys=set(), _optional_state_args=set()), 'get_weather': _InjectedArgs(state={}, store=None, runtime=None, all_injected_keys=set(), _optional_state_args=set())}, _handle_tool_errors=<function _default_handle_tool_errors at 0x000001DE7CDB5580>, _messages_ke

In [63]:
message_with_multiple_tool_calls=AIMessage(
    content="",
    tool_calls=[
        {
            "name":"fetch_real_time_info",
        "args":{"query":"小米汽车"},
        "id":"tool_call_id",
        "type":"tool_call"
        },
        {
            "name":"get_weather",
            "args":{"location":"beijing"},
            "id":"tool_call_id1",
            "type":"tool_call"
        }
    ]
)

config = {CONF: {CONFIG_KEY_RUNTIME: Runtime()}}
tool_node.invoke({"messages":[message_with_multiple_tool_calls]},config)

query:小米汽车


{'messages': [ToolMessage(content='{"latitude": 52.52, "longitude": 13.419998, "generationtime_ms": 0.0017881393432617188, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 38.0}', name='fetch_real_time_info', tool_call_id='tool_call_id'),
  ToolMessage(content='北京的温度是16度，天气晴朗。', name='get_weather', tool_call_id='tool_call_id1')]}

In [64]:
tools

[StructuredTool(name='fetch_real_time_info', description='get real-time Internet information', args_schema=<class 'langchain_core.utils.pydantic.fetch_real_time_info'>, func=<function fetch_real_time_info at 0x000001DE7FABA160>),
 StructuredTool(name='get_weather', description='call to get the current weather.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001DE7FABB880>)]

In [65]:
model_with_tools=llm.bind_tools(tools)


In [66]:
model_with_tools.kwargs

{'tools': [{'type': 'function',
   'function': {'name': 'fetch_real_time_info',
    'description': 'get real-time Internet information',
    'parameters': {'properties': {'query': {}},
     'required': ['query'],
     'type': 'object'}}},
  {'type': 'function',
   'function': {'name': 'get_weather',
    'description': 'call to get the current weather.',
    'parameters': {'properties': {'location': {}},
     'required': ['location'],
     'type': 'object'}}}]}

In [67]:
model_with_tools.invoke("小米汽车").tool_calls

[{'name': 'fetch_real_time_info',
  'args': {'query': '小米汽车 最新消息 2025'},
  'id': 'call_00_3VX9IfXIiKKRbQkFIL6O9782',
  'type': 'tool_call'}]

In [68]:
model_with_tools.invoke("beijing").tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Beijing'},
  'id': 'call_00_Y0alBFD0Gis3mRuCWRsI5949',
  'type': 'tool_call'}]

In [73]:
tool_node.invoke({"messages":[model_with_tools.invoke("小米汽车")]},config)

query:小米汽车 最新消息 2025


{'messages': [ToolMessage(content='{"latitude": 52.52, "longitude": 13.419998, "generationtime_ms": 0.0015497207641601562, "utc_offset_seconds": 0, "timezone": "GMT", "timezone_abbreviation": "GMT", "elevation": 38.0}', name='fetch_real_time_info', tool_call_id='call_00_2KPRHE3JEF1riX9nvkag6758')]}

In [72]:
tool_node.invoke({"messages":[model_with_tools.invoke("北京现在多少度")]},config)

{'messages': [ToolMessage(content='不好意思，并未查询到具体的天气信息', name='get_weather', tool_call_id='call_00_4jG4P8eaAmMNBnpWx6vj8285')]}

In [78]:
tool_node.invoke({"messages":[model_with_tools.invoke("beijing 现在多少度")]},config)

{'messages': [ToolMessage(content='北京的温度是16度，天气晴朗。', name='get_weather', tool_call_id='call_00_fAr2FempqnGjo529jiMS9505')]}

In [75]:
tool_node.invoke({"messages":[model_with_tools.invoke("shanghai现在多少度")]},config)

{'messages': [ToolMessage(content='上海的温度是30度，天气多云', name='get_weather', tool_call_id='call_00_uQBbmlQz3fkD0xQehtik6310')]}